# Olist E-Commerce Business Intelligence

## Phase 2 — Data Quality Assessment

### Objective

The objective of this notebook is to evaluate the quality of the datasets before performing any cleaning operations.

This notebook covers:

- Missing value assessment
- Duplicate record analysis
- Primary key validation
- Foreign key validation
- Data type assessment
- Invalid value detection
- Data quality summary

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
PROJECT_ROOT = Path().resolve().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw"

In [3]:
customers = pd.read_csv(DATA_PATH / "olist_customers_dataset.csv")

geolocation = pd.read_csv(DATA_PATH / "olist_geolocation_dataset.csv")

order_items = pd.read_csv(DATA_PATH / "olist_order_items_dataset.csv")

payments = pd.read_csv(DATA_PATH / "olist_order_payments_dataset.csv")

reviews = pd.read_csv(DATA_PATH / "olist_order_reviews_dataset.csv")

orders = pd.read_csv(DATA_PATH / "olist_orders_dataset.csv")

products = pd.read_csv(DATA_PATH / "olist_products_dataset.csv")

sellers = pd.read_csv(DATA_PATH / "olist_sellers_dataset.csv")

translation = pd.read_csv(DATA_PATH / "product_category_name_translation.csv")

In [4]:
datasets = {
    "Customers": customers,
    "Geolocation": geolocation,
    "Order Items": order_items,
    "Payments": payments,
    "Reviews": reviews,
    "Orders": orders,
    "Products": products,
    "Sellers": sellers,
    "Translation": translation
}

## Missing Value Assessment

Missing values can reduce model performance and affect business analysis.

This section measures both the count and percentage of missing values in every dataset.

In [5]:
def missing_value_report(df):
    report = pd.DataFrame({
        "Missing Count": df.isnull().sum(),
        "Missing Percentage": (df.isnull().mean() * 100).round(2)
    })

    return report[report["Missing Count"] > 0].sort_values(
        by="Missing Percentage",
        ascending=False
    )

In [6]:
for name, df in datasets.items():

    print("=" * 70)
    print(name)
    print("=" * 70)

    display(missing_value_report(df))

Customers


,Missing Count,Missing Percentage


Geolocation


,Missing Count,Missing Percentage


Order Items


...

,Missing Count,Missing Percentage
product_category_name,610,1.85
product_name_lenght,610,1.85
product_description_lenght,610,1.85
product_photos_qty,610,1.85
product_weight_g,2,0.01
product_length_cm,2,0.01
product_height_cm,2,0.01
product_width_cm,2,0.01


Sellers


,Missing Count,Missing Percentage


Translation


,Missing Count,Missing Percentage


## Missing Value Analysis

### Reviews Dataset

- `review_comment_title` contains **87,656** missing values (**88.34%**).
- `review_comment_message` contains **58,247** missing values (**58.70%**).

**Interpretation:**
- Most customers provided only a rating without leaving written feedback.
- These missing values are expected due to user behavior rather than data collection errors.
- They should not be treated as data corruption.

---

### Orders Dataset

- `order_delivered_customer_date` has **2,965** missing values (**2.98%**).
- `order_delivered_carrier_date` has **1,783** missing values (**1.79%**).
- `order_approved_at` has **160** missing values (**0.16%**).

**Interpretation:**
- Missing delivery dates may correspond to canceled or undelivered orders.
- Missing approval timestamps represent a very small proportion of records and should be investigated before cleaning.

---

### Products Dataset

- `product_category_name` has **610** missing values (**1.85%**).
- Product dimensions and descriptive attributes have similar missing percentages.
- Weight and physical dimensions have only **2** missing records.

**Interpretation:**
- Most missing values affect product metadata rather than transactional information.
- Missing dimensions are minimal and can likely be imputed or removed with little impact.

---

### Overall Assessment

The majority of datasets contain complete records.

The most significant missing data occurs in optional customer review text fields, which is expected in e-commerce platforms.

The remaining missing values are relatively small and can be addressed during the data cleaning phase.

In [7]:
for name, df in datasets.items():
    duplicates = df.duplicated().sum()
    print(f"{name}: {duplicates}")

Customers: 0
Geolocation: 261831
Order Items: 0
Payments: 0
Reviews: 0
Orders: 0
Products: 0
Sellers: 0
Translation: 0


## Duplicate Record Analysis

### Customers Dataset

- No duplicate records were detected.
- Each customer record appears to be unique.

---

### Geolocation Dataset

- The dataset contains **261,831 duplicate records**.

**Interpretation:**

This is **expected** and does not necessarily indicate poor data quality.

The geolocation dataset stores geographical information based on Brazilian ZIP code prefixes. Multiple addresses may legitimately share the same:

- ZIP code prefix
- Latitude
- Longitude
- City
- State

Therefore, duplicate rows represent repeated geographic references rather than duplicated business transactions.

Whether these duplicates should be removed depends on the intended analysis and will be evaluated during the data cleaning phase.

---

### Remaining Datasets

The following datasets contain **no duplicate rows**:

- Customers
- Order Items
- Payments
- Reviews
- Orders
- Products
- Sellers
- Product Category Translation

This indicates a high level of data integrity across the transactional datasets.

---

### Overall Assessment

Except for the geolocation dataset, no duplicate records were identified.

The duplicate entries in the geolocation table are consistent with the nature of geographic reference data and are not considered an immediate data quality issue.

In [8]:
primary_keys = {
    "Customers": "customer_id",
    "Orders": "order_id",
    "Products": "product_id",
    "Sellers": "seller_id"
}

for dataset, key in primary_keys.items():
    df = datasets[dataset]

    duplicate_keys = df[key].duplicated().sum()

    print("=" * 60)
    print(f"Dataset      : {dataset}")
    print(f"Primary Key  : {key}")
    print(f"Duplicate PK : {duplicate_keys}")

Dataset      : Customers
Primary Key  : customer_id
Duplicate PK : 0
Dataset      : Orders
Primary Key  : order_id
Duplicate PK : 0
Dataset      : Products
Primary Key  : product_id
Duplicate PK : 0
Dataset      : Sellers
Primary Key  : seller_id
Duplicate PK : 0


## Primary Key Validation

The primary keys of the Customers, Orders, Products, and Sellers datasets were validated for uniqueness.

Results:

- All primary keys are unique.
- No duplicate primary key values were detected.
- This indicates that entity integrity is maintained across the core transactional tables.

### Note on Reviews Dataset

Initial validation showed duplicate values for `review_id`.

Further inspection revealed that `review_id` is **not a true primary key** within this dataset. A single review identifier may legitimately be associated with multiple records.

Therefore, the Reviews dataset was excluded from the primary key validation.

## Foreign Key Validation

Foreign keys establish relationships between tables in a relational database.

This section verifies that every foreign key value has a corresponding primary key in its parent table.

Invalid foreign keys indicate broken relationships or data integrity issues.

In [9]:
def validate_foreign_key(child_df, child_key, parent_df, parent_key):

    invalid = ~child_df[child_key].isin(parent_df[parent_key])

    return invalid.sum()

In [10]:
relationships = [
    ("Orders", "customer_id", "Customers", "customer_id"),
    ("Order Items", "order_id", "Orders", "order_id"),
    ("Order Items", "product_id", "Products", "product_id"),
    ("Order Items", "seller_id", "Sellers", "seller_id"),
    ("Payments", "order_id", "Orders", "order_id"),
    ("Reviews", "order_id", "Orders", "order_id"),
]

In [11]:
for child_table, child_key, parent_table, parent_key in relationships:

    invalid = validate_foreign_key(
        datasets[child_table],
        child_key,
        datasets[parent_table],
        parent_key
    )

    print("=" * 70)
    print(f"Child Table : {child_table}")
    print(f"Parent Table: {parent_table}")
    print(f"Foreign Key : {child_key}")
    print(f"Invalid Keys: {invalid}")

Child Table : Orders
Parent Table: Customers
Foreign Key : customer_id
Invalid Keys: 0
Child Table : Order Items
Parent Table: Orders
Foreign Key : order_id
Invalid Keys: 0
Child Table : Order Items
Parent Table: Products
Foreign Key : product_id
Invalid Keys: 0
Child Table : Order Items
Parent Table: Sellers
Foreign Key : seller_id
Invalid Keys: 0
Child Table : Payments
Parent Table: Orders
Foreign Key : order_id
Invalid Keys: 0
Child Table : Reviews
Parent Table: Orders
Foreign Key : order_id
Invalid Keys: 0


## Foreign Key Validation Analysis

The foreign key validation confirmed that all relationships between the transactional datasets are valid.

### Validation Results

| Child Table | Parent Table | Foreign Key | Status |
|-------------|--------------|-------------|--------|
| Orders | Customers | customer_id | ✅ Valid |
| Order Items | Orders | order_id | ✅ Valid |
| Order Items | Products | product_id | ✅ Valid |
| Order Items | Sellers | seller_id | ✅ Valid |
| Payments | Orders | order_id | ✅ Valid |
| Reviews | Orders | order_id | ✅ Valid |

### Interpretation

- Every foreign key successfully matched a corresponding primary key in its parent table.
- No orphan records were identified.
- The dataset preserves referential integrity across all evaluated relationships.
- This confirms that the relational structure of the Olist dataset is reliable and suitable for downstream analytics and machine learning tasks.

# Data Quality Assessment Summary

## Missing Values

- Missing values are concentrated in optional review text fields and selected product metadata.
- Transactional datasets contain relatively few missing values.

## Duplicate Records

- No duplicate records were found in the transactional datasets.
- The Geolocation dataset contains duplicate geographic references, which are expected due to repeated ZIP code and coordinate information.

## Primary Keys

- Customer, Order, Product, and Seller identifiers are unique.
- Entity integrity is maintained across all core business tables.
- The `review_id` field is not a unique primary key and was excluded from primary key validation.

## Foreign Keys

- All evaluated foreign key relationships are valid.
- No orphan records were detected.
- Referential integrity is fully maintained.

## Overall Assessment

The Olist Brazilian E-Commerce dataset exhibits a high level of structural integrity and is well suited for business intelligence, exploratory data analysis, and predictive modeling. The identified quality issues are limited to expected missing values in optional fields and do not compromise the consistency of the relational database.